# Islamic Audiobook Generator
### Run each cell in order. Runtime > Change runtime type > T4 GPU

In [ ]:
# Step 1: Install dependencies
!pip install numpy==1.26.4
!pip install "librosa>=0.11.0" "s3tokenizer" "torch>=2.6.0" "torchaudio>=2.6.0" "transformers>=4.46.3" "diffusers>=0.29.0" "resemble-perth>=1.0.1" "conformer>=0.3.2" "safetensors>=0.5.3" "omegaconf" "gradio" "pymupdf" "pydub"
!pip install chatterbox-tts --no-deps

In [ ]:
# Step 2: Check GPU
!nvidia-smi

In [ ]:
# Step 3: Run the app
import os
import uuid
import fitz
import shutil
import torchaudio as ta
import gradio as gr
from pydub import AudioSegment
from chatterbox.tts import ChatterboxTTS

os.makedirs('outputs', exist_ok=True)

model = ChatterboxTTS.from_pretrained(device='cuda')

def extract_text(pdf_path):
    text = ''
    doc = fitz.open(pdf_path)
    for page in doc:
        text += page.get_text('text') or ''
    return text

def chunk_text(text, max_chars=200):
    sentences = text.replace('\n', ' ').split('. ')
    chunks, current = [], ''
    for s in sentences:
        if len(current) + len(s) < max_chars:
            current += s + '. '
        else:
            if current:
                chunks.append(current.strip())
            current = s + '. '
    if current:
        chunks.append(current.strip())
    return chunks

def generate(pdf_file, voice_file):
    job_id = str(uuid.uuid4())
    output_path = f'outputs/{job_id}_audiobook.mp3'

    text = extract_text(pdf_file)
    if not text.strip():
        return None, 'Could not extract text from PDF'

    chunks = chunk_text(text)
    audio_segments = []

    for i, chunk in enumerate(chunks):
        chunk_path = f'outputs/{job_id}_chunk_{i}.wav'
        wav = model.generate(chunk, audio_prompt_path=voice_file)
        ta.save(chunk_path, wav, model.sr)
        audio_segments.append(AudioSegment.from_wav(chunk_path))

    combined = sum(audio_segments)
    combined.export(output_path, format='mp3')

    for i in range(len(chunks)):
        os.remove(f'outputs/{job_id}_chunk_{i}.wav')

    return output_path, 'Done!'

with gr.Blocks(title='Islamic Audiobook Generator') as demo:
    gr.Markdown('# Islamic Audiobook Generator')
    gr.Markdown('Upload an Urdu Islamic PDF and a scholar voice sample (WAV, min 6 sec).')
    with gr.Row():
        pdf_input = gr.File(label='Upload Islamic Book (PDF)', file_types=['.pdf'])
        voice_input = gr.File(label='Upload Scholar Voice (WAV)', file_types=['.wav'])
    btn = gr.Button('Generate Audiobook', variant='primary')
    audio_output = gr.Audio(label='Generated Audiobook', type='filepath')
    status = gr.Textbox(label='Status')
    btn.click(fn=generate, inputs=[pdf_input, voice_input], outputs=[audio_output, status])

demo.launch(share=True)